# Archivos: guardar una imagen es tomar una decisión

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Guardar la misma imagen en distintos formatos con `cv2.imwrite` y medir qué cuesta cada decisión: cuánto pesa el archivo y cuánto se aleja del original.

## Al terminar este cuaderno van a poder:

1. Guardar imágenes con `cv2.imwrite` eligiendo el formato y sus parámetros de calidad.
2. Medir el peso en disco de un archivo y su diferencia numérica con el original.
3. Distinguir compresión con pérdida y sin pérdida a partir de evidencia, no de definiciones.
4. Localizar dónde aparecen los artefactos de compresión con un mapa de diferencias.
5. Explicar en qué situaciones se acumula la pérdida al volver a guardar un JPEG.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Formato de archivo** | La forma en que los números de la imagen se ordenan y comprimen dentro del archivo: PNG, JPEG, WEBP, BMP. |
| **Compresión sin pérdida** | El archivo se achica, pero al abrirlo se recuperan exactamente los mismos números. |
| **Compresión con pérdida** | El archivo se achica descartando información: al abrirlo, los números ya no son idénticos. |
| **Calidad** | Parámetro de los formatos con pérdida que regula cuánta información se descarta. |
| **Artefacto de compresión** | Un defecto visible que agrega el algoritmo de compresión y no estaba en la escena: bloques, halos, bandas. |
| **Generación** | Cada vuelta de abrir y volver a guardar un archivo. |

## Guardar es elegir qué conservar

**Analogía.** Guardar un texto largo se puede hacer de dos maneras. Una es comprimirlo en un archivo ZIP: ocupa menos y, al descomprimirlo, el texto vuelve completo. La otra es escribir un resumen: ocupa mucho menos, pero ya no es el texto original. PNG se comporta como el ZIP; JPEG, como el resumen.

**Bajada tecnológica.** Las redes sociales y las aplicaciones de mensajería vuelven a comprimir las fotos que se suben, para ahorrar espacio y datos. Los archivos médicos y satelitales, en cambio, se guardan sin pérdida, porque un píxel alterado puede cambiar un diagnóstico o una medición. En un proyecto de análisis de imágenes, el formato forma parte del método.

### ⚙ Configuración del entorno

In [ ]:
from pathlib import Path

import cv2
import numpy as np

from utils import mostrar_imágenes, mostrar_curvas

print(f"OpenCV {cv2.__version__} — NumPy {np.__version__}")

Preparamos dos carpetas: la de imágenes del curso, para leer, y una carpeta `salidas` dentro de este nivel, para escribir. Así los archivos que genera el cuaderno no se mezclan con los originales.

In [ ]:
CARPETA_IMÁGENES = Path("..") / "imagenes"
CARPETA_SALIDAS = Path("salidas")

# 'mkdir' crea la carpeta; 'exist_ok=True' evita el error si ya existe.
CARPETA_SALIDAS.mkdir(exist_ok=True)

ruta_globos = CARPETA_IMÁGENES / "globos.jpg"
globos_bgr = cv2.imread(filename=str(ruta_globos), flags=cv2.IMREAD_COLOR)
assert globos_bgr is not None, "No se pudo leer globos.jpg: revisen la carpeta 'imagenes'."

forma_globos = globos_bgr.shape
peso_original_kb = ruta_globos.stat().st_size / 1024
print(f"Imagen: {forma_globos} — peso del archivo original: {peso_original_kb:.0f} KB")

El archivo original es un JPEG de unos 67 KB. Lo dejamos en BGR a propósito: `cv2.imwrite` espera recibir la imagen en el orden de OpenCV. Si le pasáramos una imagen en RGB, guardaría los colores intercambiados, como vimos en `001_traduccion`.

## 1. Guardar sin pérdida: PNG

**Analogía.** Es la mudanza en la que cada objeto se envuelve y se numera: se ocupa menos lugar que llevando los muebles armados, pero al llegar todo vuelve a estar exactamente donde estaba.

**Bajada tecnológica.** PNG es el formato habitual para capturas de pantalla, gráficos, logos y cualquier imagen que se va a seguir editando, porque garantiza que ningún píxel cambie.

### ✎ Antes de ejecutar

El JPEG original pesa unos 67 KB. Si guardamos esa misma imagen como PNG, un formato que no descarta nada, ¿el archivo va a pesar más o menos? ¿Cuánto más, aproximadamente?

In [ ]:
ruta_png = CARPETA_SALIDAS / "globos.png"

# 'imwrite' guarda el arreglo en disco; el formato lo decide la extensión del nombre.
se_guardó = cv2.imwrite(filename=str(ruta_png), img=globos_bgr)

peso_png_kb = ruta_png.stat().st_size / 1024
print(f"¿Se guardó? {se_guardó} — peso como PNG: {peso_png_kb:.0f} KB")

`imwrite` devuelve `True` cuando pudo escribir el archivo. El PNG pesa **más de doce veces** lo que pesa el JPEG del que partimos. Parece contradictorio, pero tiene explicación: la imagen ya había pasado por una compresión JPEG, y ahora PNG tiene que conservar exactamente todos sus valores, incluidas las pequeñas imperfecciones que dejó esa compresión.

Para afirmar que PNG no pierde nada no alcanza con mirar: lo verificamos releyendo el archivo y comparando los números.

In [ ]:
png_releído = cv2.imread(filename=str(ruta_png), flags=cv2.IMREAD_COLOR)

# 'array_equal' responde si dos arreglos tienen exactamente los mismos valores.
es_idéntica = np.array_equal(a1=globos_bgr, a2=png_releído)
print(f"¿El PNG releído es idéntico al arreglo original? {es_idéntica}")

El resultado es `True`: los 3 millones de valores de la imagen volvieron intactos. Esa es la definición operativa de compresión sin pérdida.

## 2. Comparar formatos

Ahora repetimos el procedimiento con varios formatos. Como el paso a paso ya se entiende —guardar, releer, comparar—, lo encapsulamos en una función. Para medir la distancia al original usamos `cv2.absdiff`, que calcula la diferencia absoluta píxel a píxel.

In [ ]:
def guardar_y_medir(imagen_bgr, nombre_del_formato, nombre_de_archivo, parámetros):
    """Guarda una imagen, la vuelve a leer y mide qué costó la decisión.

    Devuelve un diccionario con el peso en KB y la diferencia media con el original.
    """
    ruta = CARPETA_SALIDAS / nombre_de_archivo
    cv2.imwrite(filename=str(ruta), img=imagen_bgr, params=parámetros)
    imagen_releída = cv2.imread(filename=str(ruta), flags=cv2.IMREAD_COLOR)
    diferencia = cv2.absdiff(src1=imagen_bgr, src2=imagen_releída)
    return {
        "formato": nombre_del_formato,
        "peso_kb": ruta.stat().st_size / 1024,
        "diferencia_media": diferencia.mean(),
    }

La función recibe la imagen y el formato, y devuelve un diccionario. Una diferencia media de 0 significa que el archivo es idéntico al original; cuanto más alta, más se alejó.

Los parámetros de cada formato no significan lo mismo, y conviene tenerlos claros antes de comparar:

| Parámetro | Rango | Qué controla | Qué cambia si lo modificamos |
|---|---|---|---|
| `IMWRITE_JPEG_QUALITY` | 0 a 100 | Cuánta información descarta JPEG | Más calidad: archivo más pesado y más fiel |
| `IMWRITE_WEBP_QUALITY` | 1 a 100 | Lo mismo, para WEBP | Igual que en JPEG |
| `IMWRITE_PNG_COMPRESSION` | 0 a 9 | Cuánto esfuerzo pone PNG en comprimir | Cambia el peso y el tiempo de guardado, **nunca** la fidelidad |

In [ ]:
formatos_a_comparar = [
    {"formato": "PNG", "archivo": "globos.png", "parámetros": [cv2.IMWRITE_PNG_COMPRESSION, 3]},
    {"formato": "BMP", "archivo": "globos.bmp", "parámetros": []},
    {"formato": "JPEG calidad 95", "archivo": "globos_95.jpg", "parámetros": [cv2.IMWRITE_JPEG_QUALITY, 95]},
    {"formato": "JPEG calidad 35", "archivo": "globos_35.jpg", "parámetros": [cv2.IMWRITE_JPEG_QUALITY, 35]},
    {"formato": "WEBP calidad 80", "archivo": "globos_80.webp", "parámetros": [cv2.IMWRITE_WEBP_QUALITY, 80]},
]

cantidad_de_formatos = len(formatos_a_comparar)
print(f"Formatos a comparar: {cantidad_de_formatos}")

Cada formato quedó descripto en un diccionario con su nombre, el archivo de salida y sus parámetros. BMP no recibe parámetros: guarda los píxeles sin comprimir.

### ✎ Antes de ejecutar

Ordenen los cinco formatos del más liviano al más pesado según lo que esperan. ¿Cuáles creen que van a tener diferencia media igual a cero?

In [ ]:
resultados_por_formato = []

for formato in formatos_a_comparar:
    resultado = guardar_y_medir(
        imagen_bgr=globos_bgr,
        nombre_del_formato=formato["formato"],
        nombre_de_archivo=formato["archivo"],
        parámetros=formato["parámetros"],
    )
    resultados_por_formato.append(resultado)

cantidad_de_resultados = len(resultados_por_formato)
print(f"Formatos guardados, releídos y medidos: {cantidad_de_resultados}")

Los cinco archivos quedaron en la carpeta `salidas`. Mostramos los resultados como una tabla: este es el primer producto del cuaderno.

In [ ]:
print(f"{'Formato':<18}{'Peso (KB)':>12}{'Diferencia media':>20}")
print(f"{'JPEG original':<18}{peso_original_kb:>12.0f}{'—':>20}")

for resultado in resultados_por_formato:
    nombre = resultado["formato"]
    peso = resultado["peso_kb"]
    diferencia = resultado["diferencia_media"]
    print(f"{nombre:<18}{peso:>12.0f}{diferencia:>20.2f}")

La tabla separa los formatos en dos grupos:

- **Sin pérdida (PNG y BMP):** diferencia media 0. BMP es el más pesado de todos, unos 3 MB, porque no comprime nada: guarda los tres bytes de cada píxel. PNG conserva lo mismo en una fracción de ese peso. El PNG de la tabla pesa menos que el de la sección 1 porque ahora le pedimos compresión 3 en lugar de dejar el valor por defecto: el archivo cambió de peso, pero sigue siendo idéntico al original.
- **Con pérdida (JPEG y WEBP):** diferencia mayor a 0. JPEG con calidad 95 se aleja muy poco del original, pero pesa casi el doble que el archivo del que partimos. JPEG con calidad 35 y WEBP con calidad 80 son los más livianos; WEBP logra un peso similar con menos diferencia.

### ✎ Para pensar

- ¿Por qué ningún formato sin pérdida puede quedar más liviano que el JPEG original de esta imagen?
- Si tuvieran que publicar 500 fotos en un sitio web, ¿qué columna de la tabla les importaría más? ¿Y si fueran radiografías?

### ✎ Completado

Agreguen dos formatos a la comparación: WEBP con calidad 30 y PNG con compresión 9. Después vuelvan a ejecutar las dos celdas anteriores.

In [ ]:
# ─── Completen los pasos faltantes ───────────────────────────────────────────
#
# TODO 1: agreguen a 'formatos_a_comparar' un diccionario para WEBP calidad 30.
# TODO 2: agreguen otro para PNG con IMWRITE_PNG_COMPRESSION igual a 9.
# TODO 3: vuelvan a ejecutar el bucle y la tabla. ¿Cambió la diferencia del PNG?
#
# formatos_a_comparar.append({"formato": ..., "archivo": ..., "parámetros": [...]})
# formatos_a_comparar.append({"formato": ..., "archivo": ..., "parámetros": [...]})
#
# ─────────────────────────────────────────────────────────────────────────────

## 3. Dónde se esconde la pérdida

**Analogía.** Un resumen de un texto conserva la idea general, pero pierde los detalles finos: una fecha exacta, un matiz, una cita. La compresión JPEG hace algo parecido con las imágenes: conserva las zonas parejas y pierde precisión donde hay cambios bruscos.

**Bajada tecnológica.** JPEG divide la imagen en bloques de 8 × 8 píxeles y, dentro de cada bloque, descarta primero los detalles más finos. Por eso las fotos muy comprimidas muestran cuadraditos y halos alrededor de los bordes.

Guardamos una versión con calidad baja para que los defectos sean visibles. El valor de `calidad_de_prueba` es el que vamos a modificar más adelante.

In [ ]:
calidad_de_prueba = 20
ruta_de_prueba = CARPETA_SALIDAS / "globos_prueba.jpg"
parámetros_de_prueba = [cv2.IMWRITE_JPEG_QUALITY, calidad_de_prueba]

cv2.imwrite(filename=str(ruta_de_prueba), img=globos_bgr, params=parámetros_de_prueba)
jpeg_de_prueba = cv2.imread(filename=str(ruta_de_prueba), flags=cv2.IMREAD_COLOR)

peso_de_prueba_kb = ruta_de_prueba.stat().st_size / 1024
print(f"JPEG con calidad {calidad_de_prueba}: {peso_de_prueba_kb:.0f} KB")

Con calidad 20 el archivo pesa menos de la mitad que el original. Para ver qué se perdió miramos un recorte chico, donde se encuentran el cielo, un globo celeste y un globo rojo.

In [ ]:
# Recortamos la misma zona en las dos versiones: filas 140 a 320, columnas 200 a 420.
recorte_original = globos_bgr[140:320, 200:420]
recorte_comprimido = jpeg_de_prueba[140:320, 200:420]

recorte_original_rgb = cv2.cvtColor(src=recorte_original, code=cv2.COLOR_BGR2RGB)
recorte_comprimido_rgb = cv2.cvtColor(src=recorte_comprimido, code=cv2.COLOR_BGR2RGB)

paneles = [
    {"imagen": recorte_original_rgb, "título": "Recorte del original"},
    {"imagen": recorte_comprimido_rgb, "título": f"JPEG calidad {calidad_de_prueba}"},
]
mostrar_imágenes(paneles, alto_de_figura=5)

En la versión comprimida los degradados suaves —el cielo y la sombra de los globos— se ven formados por cuadrados, y el borde entre los globos y el cielo tiene un halo que no estaba en el original. Son artefactos de compresión: los agregó el algoritmo, no la escena.

Mirar a simple vista sirve, pero es subjetivo. Un **mapa de diferencias** muestra en qué lugar exacto de la imagen cambiaron los valores: negro significa sin cambios, claro significa mucha diferencia.

In [ ]:
diferencia_por_canal = cv2.absdiff(src1=globos_bgr, src2=jpeg_de_prueba)
diferencia_gris = cv2.cvtColor(src=diferencia_por_canal, code=cv2.COLOR_BGR2GRAY)

# Multiplicamos por 10 para que diferencias de pocos niveles se vuelvan visibles.
diferencia_ampliada = diferencia_gris.astype(dtype=int) * 10
diferencia_ampliada = np.clip(a=diferencia_ampliada, a_min=0, a_max=255)
diferencia_ampliada = diferencia_ampliada.astype(dtype=np.uint8)

paneles = [{"imagen": diferencia_ampliada, "título": f"Diferencia × 10 (calidad {calidad_de_prueba})"}]
mostrar_imágenes(paneles, alto_de_figura=6)

Las zonas claras dibujan el contorno de los globos y las cuerdas, mientras que el cielo queda casi negro. La pérdida no se reparte de manera uniforme: se concentra en los **bordes y los detalles finos**, exactamente donde JPEG descarta primero.

### ✎ Modificación guiada

Vuelvan a la celda donde se define `calidad_de_prueba`, cambien el 20 por **5** y ejecuten esa celda y las dos siguientes. Después repitan con **60**. Antes de cada ejecución, anoten qué esperan ver en el recorte y en el mapa de diferencias.

### ✎ Para pensar

- Si el cielo casi no cambió, ¿por qué el recorte comprimido muestra cuadrados justamente en el cielo? Piensen en cómo se ve un error de un solo nivel repetido en un bloque de 8 × 8.
- ¿Qué tipo de imagen sufriría más con JPEG: una foto de un cielo despejado o la captura de pantalla de una planilla con texto chico?

## 4. ¿Se acumula la pérdida al volver a guardar?

**Analogía.** Suele decirse que guardar un JPEG muchas veces es como hacer la fotocopia de una fotocopia: cada copia empeora un poco. Es una analogía tentadora, así que en lugar de aceptarla la vamos a poner a prueba.

**Bajada tecnológica.** Una misma foto puede pasar muchas veces por compresión: se saca con el celular, se sube a una red social, alguien la descarga, la recorta y la vuelve a subir. Saber qué parte de ese recorrido degrada la imagen permite diseñar mejor un flujo de trabajo.

Preparamos dos funciones. La primera simula una edición entre guardados: achica la imagen al 90 % y la devuelve a su tamaño, como hace una plataforma que redimensiona las fotos. `resize` y sus métodos de interpolación los estudiamos en detalle en `004_encuadre`.

In [ ]:
def achicar_y_restaurar(imagen):
    """Achica la imagen al 90 % y la devuelve a su tamaño original.

    Simula una edición entre guardados. 'INTER_AREA' promedia al achicar
    e 'INTER_LINEAR' interpola al agrandar.
    """
    alto = imagen.shape[0]
    ancho = imagen.shape[1]
    ancho_reducido = int(ancho * 0.9)
    alto_reducido = int(alto * 0.9)
    imagen_reducida = cv2.resize(src=imagen, dsize=(ancho_reducido, alto_reducido), interpolation=cv2.INTER_AREA)
    imagen_restaurada = cv2.resize(src=imagen_reducida, dsize=(ancho, alto), interpolation=cv2.INTER_LINEAR)
    return imagen_restaurada

La segunda función guarda y relee la imagen varias veces seguidas, y en cada vuelta mide cuánto se alejó del original. El parámetro `editar_antes_de_guardar` decide si en cada vuelta se aplica la edición anterior.

In [ ]:
def recomprimir(imagen_bgr, calidad, generaciones, editar_antes_de_guardar):
    """Guarda y relee una imagen varias veces; devuelve la diferencia con el original en cada vuelta."""
    ruta = CARPETA_SALIDAS / "recompresion.jpg"
    parámetros = [cv2.IMWRITE_JPEG_QUALITY, calidad]
    imagen_actual = imagen_bgr.copy()
    diferencias = []
    for generación in range(generaciones):
        if editar_antes_de_guardar:
            imagen_actual = achicar_y_restaurar(imagen=imagen_actual)
        cv2.imwrite(filename=str(ruta), img=imagen_actual, params=parámetros)
        imagen_actual = cv2.imread(filename=str(ruta), flags=cv2.IMREAD_COLOR)
        diferencia = cv2.absdiff(src1=imagen_bgr, src2=imagen_actual)
        diferencias.append(diferencia.mean())
    return diferencias

### ✎ Antes de ejecutar

Vamos a guardar la imagen diez veces seguidas en JPEG con calidad 75, sin tocarla entre una vez y la siguiente. Si la analogía de la fotocopia fuera correcta, ¿qué debería pasar con la diferencia entre la primera y la décima generación?

In [ ]:
diferencias_sin_editar = recomprimir(
    imagen_bgr=globos_bgr,
    calidad=75,
    generaciones=10,
    editar_antes_de_guardar=False,
)

primera_sin_editar = diferencias_sin_editar[0]
décima_sin_editar = diferencias_sin_editar[-1]
print(f"Sin editar → generación 1: {primera_sin_editar:.2f} | generación 10: {décima_sin_editar:.2f}")

La diferencia crece muy poco: pasa de alrededor de 0,97 a 1,07 en nueve guardados más, y casi todo ese aumento ocurre en las primeras vueltas. La primera compresión se lleva la mayor parte de la pérdida; las siguientes encuentran poca información nueva para descartar. Cuando la imagen se vuelve a guardar sin cambios y con la misma calidad, **la analogía de la fotocopia se cumple solo en una medida muy chica**.

Repetimos el experimento, pero ahora editando la imagen antes de cada guardado.

In [ ]:
diferencias_editando = recomprimir(
    imagen_bgr=globos_bgr,
    calidad=75,
    generaciones=10,
    editar_antes_de_guardar=True,
)

primera_editando = diferencias_editando[0]
décima_editando = diferencias_editando[-1]
print(f"Editando → generación 1: {primera_editando:.2f} | generación 10: {décima_editando:.2f}")

Ahora la diferencia sí aumenta con cada generación. Cada edición mueve los valores de los píxeles, y la siguiente compresión encuentra información nueva para descartar. Graficamos las dos series juntas para comparar su forma.

In [ ]:
curvas = [
    {"valores": diferencias_sin_editar, "nombre": "Guardar sin editar"},
    {"valores": diferencias_editando, "nombre": "Editar y guardar"},
]
mostrar_curvas(
    curvas=curvas,
    título="Diferencia con el original a lo largo de 10 generaciones (JPEG calidad 75)",
    etiqueta_x="Generación (0 = primer guardado)",
    etiqueta_y="Diferencia media",
)

La curva de "guardar sin editar" sube apenas una décima y enseguida se aplana; la de "editar y guardar" arranca más alta y sube de manera sostenida, de 1,4 a casi 2,4. La pérdida acumulada no viene de guardar muchas veces, sino de **editar entre guardados**. Una parte de esa pérdida la produce la propia edición, que también descarta información al achicar la imagen.

### ✎ Para pensar

- Con este resultado, ¿cómo reformularían la analogía de la fotocopia para que sea correcta?
- Una persona recorta, ajusta el brillo y vuelve a guardar la misma foto en JPEG todos los días durante una semana. ¿Qué le recomendarían hacer con el archivo de trabajo?

## Actividad de producción

In [ ]:
# ─── Espacio de práctica ─────────────────────────────────────────────────────
#
# Consigna: armen la tabla de decisión para otra imagen.
#   1. Lean cards.jpg desde la carpeta 'imagenes' (en BGR).
#   2. Usen guardar_y_medir con al menos cuatro formatos o calidades distintas.
#   3. Muestren la tabla de peso y diferencia media.
#   4. Escriban en una celda de texto qué formato elegirían para: el archivo
#      de trabajo, la publicación en un sitio web y el archivo histórico.
#      Justifiquen cada elección con un número de la tabla.
#
# ─────────────────────────────────────────────────────────────────────────────

## Cierre

El formato de un archivo no es un detalle administrativo: decide qué parte de la imagen llega intacta al próximo paso del análisis.

| Formato | Pérdida | Peso en esta imagen | Uso razonable |
|---|---|---|---|
| BMP | No | El más alto | Casos puntuales sin compresión |
| PNG | No | Alto | Archivo de trabajo, gráficos, capturas |
| JPEG | Sí, regulable | Bajo | Fotos para compartir o publicar |
| WEBP | Sí, regulable | El más bajo con buena fidelidad | Publicación web |

**Limitaciones de lo que vimos.** La diferencia media es una medida simple: dos imágenes con la misma diferencia media pueden verse muy distintas, porque no tiene en cuenta cómo percibimos los bordes y las texturas. Además, trabajamos con una imagen que ya venía comprimida en JPEG, así que ningún formato puede devolverle la calidad de la escena original. Tampoco probamos imágenes con transparencia.

En el próximo cuaderno, `004_encuadre`, dejamos el disco y volvemos a la geometría de la imagen: recortar, cambiar el tamaño y espejar.